In [1]:
%run BitWise_To_Arithmetic.ipynb
%run Find_Witness.ipynb
%run Utilitaries.ipynb

from random import choices
from random import randint
import time

In [57]:
def AND(a,b):
    return a&b

def XOR(a,b):
    return a^^b

def OR(a,b):
    return a|b

def NOT(a):
    return ~a

def ADD(a,b):
    return a+b

def SUB(a,b):
    return a-b

def MUL(a,b):
    return a*b

def DIV(a,b):
    return a/b

def MOD(a,b):
    return a%b

def SHL_l(word_size,bool_print=False):
    def SHL_l(a):
        rand = randint(0,word_size-1)
        if bool_print:
            print("offset =",rand)
        return a << rand
    return SHL_l

def SHL_e(word_size):
    def SHL_e(a):
        return a << word_size
    return SHL_e

def SHL_g(word_size,bool_print=False):
    def SHL_g(a):
        rand = randint(word_size+1,2^word_size-1)
        if bool_print:
            print("offset =",rand)
        return a << rand
    return SHL_g

def SHL(word_size,l_e_g_than_word_size):
    SHL_0 = SHL_l(word_size,bool_print=True)
    SHL_1 = SHL_e(word_size)
    SHL_2 = SHL_g(word_size,bool_print=True)
    
    if l_e_g_than_word_size == 0 :
        return SHL_0
    
    if l_e_g_than_word_size == 1 :
        return SHL_1
    
    if l_e_g_than_word_size == 2 :
        return SHL_2
    
    
def SHR_l(word_size,bool_print=False):
    def SHR_l(a):
        rand = randint(0,word_size-1)
        if bool_print:
            print("offset =",rand)
        return a >> rand
    return SHR_l

def SHR_e(word_size):
    def SHR_e(a):
        return a >> word_size
    return SHR_e

def SHR_g(word_size,bool_print=False):
    def SHR_g(a):
        rand = randint(word_size+1,2^word_size-1)
        if bool_print:
            print("offset =",rand)
        return a >> rand
    return SHR_g

def SHR(word_size,l_e_g_than_word_size):
    SHR_0 = SHR_l(word_size,bool_print=True)
    SHR_1 = SHR_e(word_size)
    SHR_2 = SHR_g(word_size,bool_print=True)
    
    if l_e_g_than_word_size == 0 :
        return SHR_0
    
    if l_e_g_than_word_size == 1 :
        return SHR_1
    
    if l_e_g_than_word_size == 2 :
        return SHR_2

In [60]:
def nb_inp(function,word_size):
    if function in [NOT,SHL_l,SHL_e,SHL_g]:
        return 2^word_size
    
    if function in [DIV,MOD]:
        return 2^word_size*(2^word_size-1)
    
    return 2^(2*word_size)

In [ ]:
def nb_wit(function,word_size):
    
    if function in [NOT]:
        return 2^(word_size*3)
    
    if function in [AND,XOR,OR,MUL,SHR_l]:
        return 2^(word_size*4)
    
    if function in [ADD,SUB]:
        return 2^(word_size*3+1)
        
    if function in [SHL_e,SHL_g,SHR_e,SHR_g]:
        return 2^(word_size)
    
    if function in [SHL_l]:
        return 2^(word_size*3)
    
    return 2^(word_size*5)

In [ ]:
def size_witness(function,word_size):

    if function in [AND,XOR,OR,MUL,SHR_l]:
        return 8
    
    if function in [ADD,SUB,NOT,SHL_l]:
        return 6
    
    if function in [SHL_e,SHL_g,SHR_e,SHR_g]:
        return 2
    
    return 10

In [ ]:
def print_and_verif_nb_inputs(nb_inputs_to_test,nb_witness_to_test_by_inputs,nb_max_inp,nb_max_wit):
    
    print("Nb max of inputs :",nb_max_inp)
    print("Nb max of witness per input :",nb_max_wit)
    
    if nb_inputs_to_test>nb_max_inp:
        print("Number of inputs to test is superior to the number of total inputs")
        nb_inputs_to_test=nb_max_inp
        
    if nb_witness_to_test_by_inputs>nb_max_wit :
        print("Number of witness to test by input is superior to the number of total witness per inputs")
        nb_witness_to_test_by_inputs=nb_max_wit
    
    print("Proportion of cases tested : %d/%d (%f%%)"%(nb_inputs_to_test*nb_witness_to_test_by_inputs,nb_max_inp*nb_max_wit,float(nb_inputs_to_test*nb_witness_to_test_by_inputs)/(nb_max_inp*nb_max_wit)*100))
    
    return nb_inputs_to_test,nb_witness_to_test_by_inputs

In [ ]:
def rand_input_not_yet_choose(function,word_size,dic,nb_inputs=2):

    space_inp = range(2^word_size)
        
    inp = tuple(choices(space_inp,k=nb_inputs))

    while inp in dic or (function in [DIV,MOD] and inp[1]==0):
        inp = tuple(choices(space_inp,k=nb_inputs))
    dic[inp] = {}
    
    return inp

In [ ]:
def input_Arith(inp_BitW,w):
    inp_Arith = ()
    for i in range(len(inp_BitW)):
        inp_Arith += (inp_BitW[i],w[2*i],w[2*i+1])
    
    return inp_Arith + w[len(inp_BitW)*2:]

In [75]:
def rand_witness_not_yet_choose(function,word_size,dic):

    space_wit = EvenBits(word_size//2)
        
    w = tuple(choices(space_wit,k=size_witness(function,word_size)))
    if function in [ADD,SUB]:
        w += (randint(0,1),)

    while w in dic :
        w = tuple(choices(space_wit,k=size_witness(function,word_size)))
        if function in [ADD,SUB]:
            w += (randint(0,1),)
            
    dic[w] = {}
    
    return w

In [ ]:
def print_witness_not_valid(inp_BitW,c,w,wrong_c):
    print("\n")
    print("Witness doesn't work !")
    print("inp =",inp_BitW)
    print("w =",w)
    print("Output must be",c+[1])
    print("But C(inp,w) =",wrong_c)

In [ ]:
def print_other_valid_witness(function,inp_BitW,c,w,w2,word_size):
    print("\n")
    print("The input is", inp_BitW)
    print("The output is",c)
    print("The witness is   ", w)
    print("And should not be",w2)
    print("a =",inp_BitW[0],"=",2*w2[1]+w2[0],"= 2*a_o+a_e")
    
    if len(inp_BitW) == 2 :
        print("b =",inp_BitW[2],"=",2*w2[3]+w2[2],"= 2*b_o+b_e")
        
    if function in [AND,XOR,OR]:
        print("a_e + b_e =",w2[0] + w2[2],"=",2*w2[5]+w2[4],"= 2*e_o+e_e")
        print("a_o + b_o =",w2[1] + w2[3],"=",2*w2[7]+w2[6],"= 2*o_o+o_e") 
        print("c =",c,"=",2*w2[5]+w2[4],"= 2*c_o+c_e")
        
    if function == AND :
        print("c =",c,"=",2*w2[7]+w2[5],"= 2*o_o+e_o")
        
    if function == XOR :
        print("c =",c,"=",2*w2[6]+w2[4],"= 2*o_e+e_e")
        
    if function == OR :
        print("c_AND =",2*w2[7]+w2[5],"= 2*o_o+e_o")
        print("c_XOR =",2*w2[6]+w2[4],"= 2*o_e+e_e")
        print("c =",c,"=",2*w2[7]+w2[5]+2*w2[6]+w2[4],"= c_AND+c_XOR")
        
    if function in [ADD,SUB,MUL]:
        print("c =",c,"=",2*w2[5]+w2[4],"= 2*c_o+c_e")
        
    if function == DIV :
        print("q =",c,"=",2*w2[5]+w2[4],"= 2*q_o+q_e")
        print("r =",2*w2[7]+w2[6],"= 2*r_o+r_e")
        print("b-r-1 =",inp_BitW[1]-2*w2[7]+w2[6]-1,"=",2*w2[9]+w2[8],"= 2*{b-r-1}_o + {b-r-1}_e")
        
    if function == MOD :
        print("q =",2*w2[5]+w2[4],"= 2*q_o+q_e")
        print("r =",c,"=",2*w2[7]+w2[6],"= 2*r_o+r_e")
        print("b-r-1 =",inp_BitW[1]-2*w2[7]+w2[6]-1,"=",2*w2[9]+w2[8],"= 2*{b-r-1}_o + {b-r-1}_e")
        
    if function == ADD :
        print("k =",w2[6])
        print("a+b =",inp_BitW[0]+inp_BitW[1],"=",c+w2[6]*2^word_size,"= c+k*2^word_size")
        
    if function == SUB :
        print("b+c =",inp_BitW[1]+c,"=",inp_BitW[1]+w2[6]*2^word_size,"= a+k*2^word_size")
        
    if function == MUL :
        print("k =",2*w2[7]+w2[6],"=2*k_e+k_o")
        print("a*b =",inp_BitW[0]+inp_BitW[1],"=",c+(2*w2[7]+w2[6])*2^word_size,"= a+k*2^word_size")
        
    if function in [SHL_e,SHL_g] :
        print("c =",c,"=",2*w2[3]+w2[2],"= 2*c_o+c_e")
        
    if function == SHL_l :
        print("k =",2*w2[5]+w2[4],"=2*k_e+k_o")
        print("a*2^b =",c+(2*w2[5]+w2[4])*2^word_size,"= c+k*2^word_size")
        
        
        

In [27]:
def test_aux(function,word_size,nb_inputs_to_test,nb_witness_to_test_by_inputs,idk=None):
    
    nb_inputs = 1
    if function in [AND,OR,XOR,ADD,SUB,MUL,DIV,MOD]:
        nb_inputs = 2
        
    C_BitW = fun_to_circuit_BitWise(function,nb_inputs,word_size)
    if idk != None:
        function = idk
    
    C_Arith,dict_proof = fun_to_circuit_Arithmetic(C_BitW,word_size)
    
    nb_max_inp = nb_inp(function,word_size)
    nb_max_wit = nb_wit(function,word_size)       
    nb_inputs_to_test,nb_witness_to_test_by_inputs = print_and_verif_nb_inputs(nb_inputs_to_test,nb_witness_to_test_by_inputs,nb_max_inp,nb_max_wit)
    
    dic = {}
    
    cpt = 1
    start = time.time()
    
    is_no_pb = True
    for _ in range(nb_inputs_to_test):
        
        inp_BitW = rand_input_not_yet_choose(function,word_size,dic,nb_inputs)
        
        c = C_BitW.evaluate(inp_BitW)

        T_BitW = C_BitW.trace(inp_BitW)

        w = witness(T_BitW,dict_proof,word_size)
        
        inp_Arith = input_Arith(inp_BitW,w)
        
        if C_Arith.evaluate(inp_Arith) != c+[True]:
            print_witness_not_valid(inp_BitW,c,w,C_Arith.evaluate(inp_Arith))
        
        for _ in range(nb_witness_to_test_by_inputs):
            end = time.time()
            if end - start >0.001 :
                drawProgressBar(float(cpt)/(nb_inputs_to_test*nb_witness_to_test_by_inputs),barLen=100,print_100=True)
                start = end
            cpt += 1
            
            w2 = rand_witness_not_yet_choose(function,word_size,dic[inp_BitW])
            inp_Arith = input_Arith(inp_BitW,w2)

            if C_Arith.evaluate(inp_Arith) == c+[True]:               
                if w2!=w : #and (function not in [SHL] or w2[0:5]+w2[7:]!= w[0:5]+w[7:]):
                    is_no_pb = False
                    print_other_valid_witness(function,inp_BitW,c[0],w,w2,word_size) 
                        
    drawProgressBar(1,barLen=100,print_100=True)
    return is_no_pb

In [ ]:
def test(function,word_size,nb_inputs_to_test,nb_witness_to_test_by_inputs):
    
    if function not in [SHL,SHR] :
        is_no_pb = test_aux(function,word_size,nb_inputs_to_test,nb_witness_to_test_by_inputs)
    else :
        if function == SHL :
            function_l = SHL_l
            function_e = SHL_e
            function_g = SHL_g
            
        elif function == SHR :
            function_l = SHR_l
            function_e = SHR_e
            function_g = SHR_g
            
        print("\n")
        print("Test for offset < word_size")
        is_no_pb = test_aux(function(word_size,0),word_size,nb_inputs_to_test,nb_witness_to_test_by_inputs,function_l)
        
        print("\n")
        print("Test for offset == word_size")
        is_no_pb &= test_aux(function(word_size,1),word_size,nb_inputs_to_test,nb_witness_to_test_by_inputs,function_e)
        
        print("\n")
        print("Test for offset > word_size")
        is_no_pb &= test_aux(function(word_size,2),word_size,nb_inputs_to_test,nb_witness_to_test_by_inputs,function_g)
        
    return is_no_pb

In [9]:
def random_function(r,word_size):
    
    SHL_0 = SHL_l(word_size)
    SHL_1 = SHL_e(word_size)
    SHL_2 = SHL_g(word_size)
    
    SHR_0 = SHR_l(word_size)
    SHR_1 = SHR_e(word_size)
    SHR_2 = SHR_g(word_size)
    
    nb_gates = len(r)
    
    def fun(List):
        L = List.copy()
        nb_inp = len(L)
        for i in range(nb_gates):
            a = L[r[i][1]]
            b = L[r[i][2]]
            if a == -1:
                a = randint(0,2^word_size-1)
            if b == -1:
                b = randint(0,2^word_size-1)
                
            if(r[i][0]==0):
                L.append(AND(a,b))
                
            elif(r[i][0]==1):
                L.append(XOR(a,b))
                
            elif(r[i][0]==2):
                L.append(OR(a,b))
                
            elif(r[i][0]==3):
                L.append(NOT(a))
                
            elif(r[i][0]==4):
                L.append(ADD(a,b))
                
            elif(r[i][0]==5):
                L.append(SUB(a,b))
                
            elif(r[i][0]==6):
                L.append(MUL(a,b))
                
            elif(r[i][0]==13):
                if b == 0:
                    b = randint(1,2^word_size-1)
                L.append(DIV(a,b))
                
            elif(r[i][0]==14):
                if b == 0:
                    b = randint(1,2^word_size-1)
                L.append(MOD(a,b))
                
            elif(r[i][0]==7):
                L.append(SHL_0(a))
                         
            elif(r[i][0]==8):
                L.append(SHL_1(a))
                         
            elif(r[i][0]==9):
                L.append(SHL_2(a))
                         
            elif(r[i][0]==10):
                L.append(SHR_0(a))
                         
            elif(r[i][0]==11):
                L.append(SHR_1(a))
                         
            elif(r[i][0]==12):
                L.append(SHR_2(a))
                
        return L[nb_gates+nb_inp-1]
    return fun

In [ ]:
nb_gates = 10

nb_inputs = 3

r = []

for i in range(nb_gates):
    gate = randint(0,14)
    inp1 = randint(0,nb_inputs-1+i)
    if gate in [3]+[9..14]:
        r.append((gate,inp1))
    else : 
        inp2 = randint(0,nb_inputs-1+i)
        r.append((gate,inp1,inp2))